# 04 - Invoke Azure Function and View Logs

This notebook invokes the Speech Function with function-key authentication, verifies that the function can call the Speech endpoint with managed identity, and then queries recent logs.

## Step 1 - Load environment and helpers

In [9]:
import json
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env", override=True)

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

resource_group = os.getenv("AZURE_RESOURCE_GROUP")
if not resource_group:
    raise RuntimeError("Missing AZURE_RESOURCE_GROUP in ../env/.env")

print(f"Resource group: {resource_group}")

Resource group: rg-speech01-foundry


## Step 2 - Load Function values from env

This notebook expects function metadata and the function key to already exist in ../env/.env from Notebook 2.

In [2]:
function_app_name = os.getenv("AZURE_FUNCTION_APP_NAME", "").strip()
function_url = os.getenv("AZURE_FUNCTION_URL", "").strip()
app_insights_name = os.getenv("AZURE_APP_INSIGHTS_NAME", "").strip()
function_key = os.getenv("AZURE_FUNCTION_KEY", "").strip()

if not function_app_name or not function_url or not app_insights_name or not function_key:
    raise RuntimeError(
        "Missing function metadata or function key in ../env/.env. Rerun Notebook 2 to refresh the key and then rerun this notebook."
    )

print(f"Function app: {function_app_name}")
print(f"Function URL: {function_url}")
print(f"App Insights: {app_insights_name}")
print("Function key: loaded")

Function app: func-speech01-6m4wo
Function URL: https://func-speech01-6m4wo.azurewebsites.net/api/speech-roundtrip
App Insights: appi-speech01-6m4wo
Function key: loaded


## Step 3 - Invoke function with function key

A successful response should include `auth_mode: managed_identity`, the Speech endpoint, a request id, and the synthesized audio byte count.

In [3]:
payload = {
    "text": "Managed identity lets this function call the Azure Speech endpoint without storing secrets."
}


def invoke_with_key(key_to_use: str):
    return requests.post(
        function_url,
        params={"code": key_to_use},
        headers={
            "Content-Type": "application/json",
            "x-functions-key": key_to_use,
        },
        json=payload,
        timeout=60,
    )


try:
    resp = invoke_with_key(function_key)
except requests.RequestException as ex:
    raise RuntimeError(f"Function invocation failed before receiving a response: {ex}") from ex

if resp.status_code == 401:
    raise RuntimeError(
        "Function invocation returned HTTP 401 (stale or invalid function key). "
        "Rerun Notebook 2 to refresh AZURE_FUNCTION_KEY in ../env/.env, then rerun this step."
    )

if resp.status_code < 200 or resp.status_code >= 300:
    raise RuntimeError(
        f"Function invocation failed with HTTP {resp.status_code}.\n"
        f"Response body: {resp.text}"
    )

print(f"HTTP status: {resp.status_code}")
try:
    print(json.dumps(resp.json(), indent=2))
except Exception:
    print(resp.text)

HTTP status: 200
{
  "input_text": "Managed identity lets this function call the Azure Speech endpoint without storing secrets.",
  "voice": "en-US-AvaMultilingualNeural",
  "speech_endpoint": "https://aispeech016m4wo.cognitiveservices.azure.com",
  "auth_mode": "managed_identity",
  "request_id": "",
  "content_type": "audio/x-wav",
  "audio_bytes": 194044,
  "message": "Managed identity successfully called the Speech endpoint."
}


## Step 4 - Query recent function traces

In [8]:
from datetime import timedelta

from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
if not subscription_id:
    raise RuntimeError(
        "Missing AZURE_SUBSCRIPTION_ID in ../env/.env. "
        "Add it, then rerun this step."
    )

if not app_insights_name:
    raise RuntimeError("App Insights component not found in this resource group.")

kql = "traces | where timestamp > ago(30m) | order by timestamp desc | take 25"
resource_id = (
    f"/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Insights/components/{app_insights_name}"
)

credential = DefaultAzureCredential()
logs_client = LogsQueryClient(credential)
query_result = logs_client.query_resource(
    resource_id=resource_id,
    query=kql,
    timespan=timedelta(minutes=30),
)

if query_result.status != LogsQueryStatus.SUCCESS:
    details = []
    for err in query_result.partial_error.details or []:
        details.append(str(err))
    raise RuntimeError(
        "Failed to query Application Insights traces with Azure Monitor Query SDK.\n"
        f"error: {query_result.partial_error}\n"
        f"details: {' | '.join(details)}"
    )

if not query_result.tables:
    print("No traces found in the last 30 minutes.")
else:
    table = query_result.tables[0]
    columns = [getattr(c, "name", str(c)) for c in table.columns]
    print(" | ".join(columns))
    print("-" * 80)
    for row in table.rows:
        print(" | ".join(str(v) for v in row))

timestamp | message | severityLevel | itemType | customDimensions | customMeasurements | operation_Name | operation_Id | operation_ParentId | operation_SyntheticSource | session_Id | user_Id | user_AuthenticatedId | user_AccountId | application_Version | client_Type | client_Model | client_OS | client_IP | client_City | client_StateOrProvince | client_CountryOrRegion | client_Browser | cloud_RoleName | cloud_RoleInstance | appId | appName | iKey | sdkVersion | itemId | itemCount | _ResourceId
--------------------------------------------------------------------------------
2026-06-08 11:12:13.081369+00:00 | ConcurrencyOptions
{
  "DynamicConcurrencyEnabled": false,
  "MaximumFunctionConcurrency": 500,
  "CPUThreshold": 0.8,
  "SnapshotPersistenceEnabled": true
} | 1 | trace | {"ProcessId":"35","HostInstanceId":"ee85eb7d-246f-4814-a668-f29199c35124","prop__{OriginalFormat}":"ConcurrencyOptions\n{\n  \"DynamicConcurrencyEnabled\": false,\n  \"MaximumFunctionConcurrency\": 500,\n  \"CPUThr